# 분할 archive 준비: 합성 검사와 선택적 Drive 진입점

기본 실행은 네트워크 데이터셋 없이 tiny split tar.gz를 생성·선택 추출·매칭합니다. 실제 Colab 업로드·Drive API·GPU는 미검증입니다. M0/M1 전체 완료나 train_ready를 선언하지 않습니다. 최신 로컬 커밋의 소스 ZIP을 사용하고 아래 실제 Drive 셀은 기본 비활성입니다.

소스 ZIP은 64MiB/추출 크기 128MiB 상한으로 코드만 풀며 원본 데이터 archive와 다른 입력입니다. [준비 계약](../docs/ARCHIVE_PREPARATION.md), [Colab 안내](../docs/COLAB_SETUP.md)를 함께 확인하세요.


In [ ]:
import sys, os, shutil
print({"python":sys.version.split()[0],"free_disk_bytes":shutil.disk_usage(".").free})


In [ ]:
from google.colab import files
from pathlib import Path, PurePosixPath
import io, stat, tempfile, zipfile
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("소스 ZIP 한 개를 선택하세요.")
zip_name, zip_bytes = next(iter(uploaded.items()))
if not zip_name.endswith(".zip") or len(zip_bytes) > 64*1024**2:
    raise ValueError("64MiB 이하의 소스 ZIP 필요")
extract_root = Path(tempfile.mkdtemp(prefix="rewind_source_", dir="/content"))
excluded = {".git", ".venv", "data", "outputs", "checkpoints", ".aws", ".codex", ".agents"}
with zipfile.ZipFile(io.BytesIO(zip_bytes)) as archive:
    members = archive.infolist()
    if sum(m.file_size for m in members) > 128*1024**2:
        raise ValueError("소스 ZIP의 해제 크기 상한 초과")
    for member in members:
        p = PurePosixPath(member.filename)
        if p.is_absolute() or ".." in p.parts or any(":" in part for part in p.parts) or stat.S_ISLNK(member.external_attr >> 16):
            raise ValueError("잘못된 ZIP member 경로")
        if not any(part in excluded for part in p.parts):
            archive.extract(member, extract_root)
roots = [p.parent.parent for p in extract_root.rglob("rewind_archive/__init__.py")]
if len(roots) != 1:
    raise ValueError("rewind_archive 패키지가 하나인 프로젝트 ZIP 필요")
PROJECT_ROOT = roots[0]
print("source:", PROJECT_ROOT)
stamp = PROJECT_ROOT / "SOURCE_REVISION.txt"
print("revision:", stamp.read_text().strip() if stamp.exists() else "unversioned; code hash로 기록")
os.chdir(PROJECT_ROOT)
sys.path.insert(0,str(PROJECT_ROOT))


In [ ]:
import subprocess
for test_pattern in ("test_archive.py", "test_drive_metadata.py"):
    subprocess.run([sys.executable,"-m","unittest","discover","-s","tests","-p",test_pattern,"-v"],cwd=PROJECT_ROOT,check=True)


## Tiny 합성 준비

분할 조각은 gzip의 중간에서 나뉘며 하나씩 독립 해제하지 않습니다. 결과와 준비 중 임시 파일은 `/content` 아래 source 프로젝트 outputs에 생성됩니다. Runtime이 삭제되면 소실되므로 필요한 결과·manifest는 별도 영구 보관합니다.


In [ ]:
import datetime, uuid, json
RUN_ROOT = PROJECT_ROOT/"outputs"/("archive_synthetic_"+datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%d_%H%M%S")+"_"+uuid.uuid4().hex[:8])
RUN_ROOT.mkdir(parents=True,exist_ok=False)
subprocess.run([sys.executable,"-m","rewind_archive.fixture","--output",str(RUN_ROOT/"fixture")],cwd=PROJECT_ROOT,check=True)
for kind in ("rgb","pose"):
    subprocess.run([sys.executable,"-m","rewind_archive","extract","--manifest",str(RUN_ROOT/"fixture"/f"{kind}_manifest.json"),
        "--limits",str(PROJECT_ROOT/"configs/archive_limits_smoke.json"),"--include","Fixture/Day/person/avatar/001/Motion__1_/*",
        "--output",str(RUN_ROOT/kind)],cwd=PROJECT_ROOT,check=True)
    subprocess.run([sys.executable,"-m","rewind_archive","verify","--extraction",str(RUN_ROOT/kind),
        "--manifest",str(RUN_ROOT/"fixture"/f"{kind}_manifest.json")],cwd=PROJECT_ROOT,check=True)
subprocess.run([sys.executable,"-m","rewind_archive","match","--extraction",str(RUN_ROOT/"rgb"),"--extraction",str(RUN_ROOT/"pose"),
    "--output",str(RUN_ROOT/"matching.json")],cwd=PROJECT_ROOT,check=True)
matching=json.loads((RUN_ROOT/"matching.json").read_text())
print({"matched_triplet_count":matching["matched_triplet_count"],"incomplete_frame_count":matching["incomplete_frame_count"],"train_ready":matching["train_ready"]})
assert matching["matched_triplet_count"]==4 and matching["train_ready"] is False


## 사용자 입력: 한 archive의 000-prefix

다음 dict에 archive_id/dataset_version/kind/공식 기대 total_parts와 순서 있는 part index/name/file_id만 입력합니다. Size/version/modifiedTime은 입력하지 않습니다. 이름은 같은 `.tar.gz` stem과 연속 000-prefix여야 합니다. File ID는 개인 입력이며 저장소에 커밋하지 않습니다.

선택 JSON이 있다면 INPUT_MODE를 `json_upload`로 바꿉니다. Metadata 셀을 활성화했을 때 작은 `drive_part_selection_v1` JSON 한 개만 업로드합니다. 직접 dict 입력은 `dict`를 사용합니다. 준비할 sequence/member glob도 별도로 지정합니다. Metadata 단계는 glob 없이 실행할 수 있지만 pilot에는 명시적 glob이 필요합니다.


In [ ]:
INPUT_MODE = "dict"  # "json_upload"도 가능
DRIVE_SELECTION = {
    "schema_version":"drive_part_selection_v1",
    "archive_id":"", "dataset_version":"", "kind":"pose", "total_parts":None,
    "parts":[]  # [{"index":0,"name":"USER_ARCHIVE.tar.gz000","file_id":"USER_FILE_ID"}]
}
SELECTED_MEMBER_GLOBS = []  # 정확한 sequence prefix + /json/*.json 등 사용자 선택
METADATA_RESULT = None  # 입력 셀을 재실행하면 이전 성공 결과를 폐기한다.


## 1. Metadata 확인과 manifest 생성 — 기본 비활성

RUN_DRIVE_METADATA를 True로 바꾼 뒤 실행합니다. 선택 규격을 먼저 검사하고 사용자가 인증합니다. 명시한 ID의 metadata만 조회하며 media·전체 폴더검색·자동 재시도는 하지 않습니다. 성공 시 실제 size/version/modifiedTime을 포함한 `archive_manifest.json`이 생성됩니다. 실패 시 manifest 없이 sanitized `metadata_report.json`만 남고 pilot은 차단됩니다.

Metadata 기본 상한은 파일8/요청8, 응답당16KiB/누적128KiB, 협력적30초/요청5초입니다. Source revision·code hash·환경·stats를 기록합니다. 인증과 credential refresh는 단계 밖에서 수행하며 token을 출력·저장하지 않습니다. 응답 body만 집계하고 HTTP/TLS/인증/transport 선수신·재전송을 제외합니다. DNS/auth/OS 전체 hard deadline이나 프로세스 RSS 상한은 아닙니다.

Metadata 재실행 시작 시 기존 result/session을 폐기합니다. 선택 입력 변경 후에는 metadata를 다시 확인합니다. 로컬의 실제 requests/urllib3/google-auth stack은 익명 credential+loopback으로 검사했지만 **실제 Colab/Drive 인증/API는 미검증**입니다.


In [ ]:
RUN_DRIVE_METADATA = False
METADATA_RESULT = None
METADATA_BUNDLE = None
PILOT_RUN_ROOT = None
if globals().get("DRIVE_SESSION") is not None:
    DRIVE_SESSION.close()
DRIVE_SESSION = None
if RUN_DRIVE_METADATA:
    from rewind_archive.__main__ import read_bounded_json
    from rewind_archive.contracts import PreparationError
    from rewind_archive.drive_metadata import MetadataLimits, validate_selection, build_drive_manifest, save_metadata_bundle
    from rewind_data.audit import read_json
    if INPUT_MODE == "json_upload":
        from google.colab import files
        selection_upload = files.upload()
        if len(selection_upload)!=1 or len(next(iter(selection_upload.values())))>128*1024:
            DRIVE_SELECTION = None
        else:
            try:
                DRIVE_SELECTION = read_json(next(iter(selection_upload.values())))
            except (ValueError, UnicodeError):
                DRIVE_SELECTION = None
    elif INPUT_MODE != "dict":
        DRIVE_SELECTION = None
    metadata_limits = MetadataLimits.from_dict(read_bounded_json(PROJECT_ROOT/"configs/drive_metadata_limits_smoke.json"))
    PILOT_RUN_ROOT = PROJECT_ROOT/"outputs"/("drive_pilot_"+uuid.uuid4().hex)
    valid_input = True
    try:
        validate_selection(DRIVE_SELECTION)
    except (PreparationError, ValueError):
        valid_input = False
    if valid_input:
        from google.colab import auth
        import google.auth
        from google.auth.transport.requests import AuthorizedSession, Request
        from requests.adapters import HTTPAdapter
        import urllib3
        if not callable(getattr(urllib3.response.HTTPResponse,"read1",None)):
            raise RuntimeError("urllib3 raw.read1 API가 필요합니다. 환경을 확인하세요.")
        auth.authenticate_user()
        credentials,_ = google.auth.default(scopes=["https://www.googleapis.com/auth/drive.readonly"])
        try:
            credentials.refresh(Request())
        except Exception:
            raise RuntimeError("인증 초기화 실패. 인증 환경을 확인하세요.") from None
        DRIVE_SESSION = AuthorizedSession(credentials,max_refresh_attempts=0)
        DRIVE_SESSION.mount("https://",HTTPAdapter(max_retries=0))
    METADATA_RESULT = build_drive_manifest(DRIVE_SELECTION,DRIVE_SESSION,metadata_limits)
    METADATA_BUNDLE = save_metadata_bundle(METADATA_RESULT,PILOT_RUN_ROOT/"metadata")
    r=METADATA_RESULT["report"]
    print({"manifest_ready":r["manifest_ready"],"stop_reason":r["stop_reason"],"stats":r["stats"],
        "media_requested_by_builder":r["media_requested_by_builder"],"train_ready":False,"output":str(METADATA_BUNDLE)})
else:
    print("실제 Drive 인증/metadata 조회를 실행하지 않았습니다.")


## 2. 별도의 bounded pilot 읽기·verify·보고 — 기본 비활성

Metadata 결과를 읽고 기존 smoke 상한을 확인한 후 RUN_DRIVE_PILOT을 True로 바꿉니다. 한 archive만으로 실행하며 RGB/pose pair 완성과 학습은 별도입니다. 자동 범위 확대·재시도·학습·Drive 쓰기는 하지 않습니다. 선택 입력이 바뀌었거나 metadata가 실패/미실행이면 이전 성공 결과로 진행할 수 없습니다.

기존 상한은 media 및 전후 metadata body 합8MiB, 협력적60초, IO64KiB/해제32MiB, 파일4MiB/32개, logical disk16MiB, metadata64KiB/header1000/range128입니다. 첫 metadata 단계의128KiB/30초와는 별도이며 report가 두 단계 stats/limits를 구분합니다. 현재 temp 결과와 작은 manifest/report bundle은 `/content` 런타임 종료 시 소실될 수 있어 필요한 결과는 사용자가 영구 보관합니다.

Drive 응답은 Content-Length가 있는 metadata와 정확한206 media, raw.read1이 필요합니다. CRC/원본변경/unsafe 입력은 unusable로 기록합니다. 같은 runtime의 verify는 개별 hash를 검사하며 실제 관절·단위·좌표/calibration·timestamp·split/원본motion·stereo 내용 정합성을 증명하지 않습니다.


In [ ]:
RUN_DRIVE_PILOT = False
PILOT_REPORT = None
if RUN_DRIVE_PILOT:
    from rewind_archive.contracts import Limits, digest
    from rewind_archive.drive_metadata import validate_selection
    from rewind_archive.pilot import run_metadata_pilot
    from rewind_archive.__main__ import read_bounded_json
    if METADATA_RESULT is None or DRIVE_SESSION is None or PILOT_RUN_ROOT is None:
        raise RuntimeError("성공한 metadata 단계를 먼저 실행하세요.")
    validate_selection(DRIVE_SELECTION)
    if METADATA_RESULT["report"]["selection_sha256"] != digest(DRIVE_SELECTION):
        raise RuntimeError("선택 입력이 바뀌었습니다. Metadata를 다시 확인하세요.")
    if not SELECTED_MEMBER_GLOBS:
        raise RuntimeError("선택할 sequence/member glob을 명시하세요.")
    extraction_limits = Limits.from_dict(read_bounded_json(PROJECT_ROOT/"configs/archive_limits_smoke.json"))
    print("extraction_limits:",json.dumps(extraction_limits.__dict__,indent=2))
    PILOT_REPORT = run_metadata_pilot(METADATA_RESULT,PILOT_RUN_ROOT/"pilot",SELECTED_MEMBER_GLOBS,extraction_limits,DRIVE_SESSION)
    print({k:PILOT_REPORT[k] for k in ("complete_files","verification","stats","metadata_stage_stats",
        "stop_reason","failure_class","gzip_eof_crc_valid","whole_archive_integrity_verified","train_ready","pending_verifications")})
    print("local bundle:",PILOT_RUN_ROOT)
else:
    print("실제 Drive media/pilot을 실행하지 않았습니다.")


## 결과 해석과 재사용

Metadata가 성공해도 학습 데이터 검증은 아닙니다. `metadata/metadata_report.json`, 성공 시 `metadata/archive_manifest.json`, `pilot/extraction_manifest.json`·완전 파일과 `pilot/pilot_report.json`에 source/hash/환경과 단계별 stats·중단·CRC·verify·미검증 항목을 보관합니다. 모든 train_ready는 false입니다.

같은 runtime의 추출본은 verify로 code/manifest/개별 hash를 확인해 재사용합니다. 현재 원격의 후속 변화나 전체 dataset을 증명하지 않습니다. Runtime 재생성 시 추출본이 없으면 000부터 다시 읽는 비용이 발생합니다. Matching valid와 triplet 수는 다르며 split은unknown, 원본motion은unverified입니다. 실제 데이터 연결·RGB loader·M1 baseline은 후속 범위입니다.
